In [1]:
# Load all of the libraries
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import intake

import sys
sys.path.append("..")
import fxns


%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.figsize'] = 12, 6

# easier to read displays in console
pd.set_option('display.max_columns', None)


In [2]:
# Specify the scenarios and variables of interest for scaling
exps = ['historical', 'ssp370',
            'ssp585', 'ssp126', 'ssp245',
            'ssp460' ]

# The url path that contains to the pangeo archive table of contents.
url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
dat = intake.open_esm_datastore(url)
dat = dat.df
out = (dat.loc[dat['grid_label'] == "gn"][["source_id", "experiment_id", "member_id", "variable_id",  
                                           "zstore", "table_id"]].copy())
out = out.rename(columns={"source_id": "model", "experiment_id": "experiment",
                          "member_id": "ensemble", "variable_id": "variable",
                            "zstore": "zstore", "table_id": "domain"}).copy()
out = (out.loc[out['experiment'].isin(exps)]).drop_duplicates().reset_index(drop=True).copy()
out.to_csv('pangeo_table.csv', index=False)


In [3]:

pangeo_data = pd.read_csv('pangeo_table.csv')


var_list = ['npp']

esm_list= ['CanESM5']

# esm_list = pangeo_data.model.unique().copy()

file_list = pd.DataFrame()
for esm in esm_list:
    for scn in exps:
        for var in var_list:
            # ###################################################
            # Step 1a: prep list of netcdf addresses on pangeo
            # ###################################################

            # get the relevant address
            nc_address = pangeo_data[((pangeo_data['model'] == esm))
                                    & ((pangeo_data['variable'] == var))
                                    & ((pangeo_data['experiment'] == scn))
                                    & ((pangeo_data['domain'].str.contains('mon')))].drop_duplicates().copy()

            # Keep only p1 ensemble members
            nc_address = nc_address[nc_address['ensemble'].str.contains('p1')].drop_duplicates().reset_index(drop=True).copy()
            # isolate the file list for opening to calculate the ensemble average data
            file_list = pd.concat([file_list,
                                   nc_address],
                                   ignore_index=True)




file_list

,model,experiment,ensemble,variable,zstore,domain
0,CanESM5,historical,r11i1p1f1,npp,gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical...,Lmon
1,CanESM5,historical,r13i1p1f1,npp,gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical...,Lmon
2,CanESM5,historical,r10i1p1f1,npp,gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical...,Lmon
3,CanESM5,historical,r12i1p1f1,npp,gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical...,Lmon
4,CanESM5,historical,r8i1p1f1,npp,gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical...,Lmon
...,...,...,...,...,...,...
95,CanESM5,ssp245,r10i1p1f1,npp,gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp...,Lmon
96,CanESM5,ssp245,r12i1p1f1,npp,gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp...,Lmon
97,CanESM5,ssp245,r15i1p1f1,npp,gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp...,Lmon
98,CanESM5,ssp245,r13i1p1f1,npp,gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp...,Lmon


In [4]:
# pick just a couple ensemble members for now
final_list = file_list[file_list['ensemble'].isin(['r10i1p1f1', 'r4i1p1f1', 'r11i1p1f1'])]

final_list
for index, row in final_list.iterrows():
    fxns.download_ncs(row, write_dir = 'data/raw-data2')


Fetching: gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical/r11i1p1f1/Lmon/npp/gn/v20190429/


Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more 

Fetching: gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical/r10i1p1f1/Lmon/npp/gn/v20190429/


Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more 

Fetching: gs://cmip6/CMIP6/CMIP/CCCma/CanESM5/historical/r4i1p1f1/Lmon/npp/gn/v20190429/


...

Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more 

Fetching: gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp245/r11i1p1f1/Lmon/npp/gn/v20190429/


Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more 

Fetching: gs://cmip6/CMIP6/ScenarioMIP/CCCma/CanESM5/ssp245/r10i1p1f1/Lmon/npp/gn/v20190429/


Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more 